# [6.1] SAE Variants - Solutions

This notebook runs the solved SAE-variant contracts and then displays the report-backed Pythia-70M TopK SAE signature result. Keep the claim boundary in view: this is a local SAE mechanics preflight, not a full semantic feature benchmark.

<details>
<summary>Expected output</summary>

The local tests should all print pass messages, and the final table/plots should match the committed `verification_report.json` metrics.

</details>

<details>
<summary>Help - why these controls matter</summary>

Sparse features are easy to overread. Reconstruction, density, planted recovery, held-out AUC, negative controls, and steering controls each remove one easy failure mode.

</details>


In [ ]:
import sys
from pathlib import Path

chapter = "chapter6_sparse_feature_methods"
section = "part1_sae_variants"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_sae_variants.tests as tests
import part1_sae_variants.solutions as solutions

relu_l1_encode = solutions.relu_l1_encode
topk_encode = solutions.topk_encode
gated_encode = solutions.gated_encode
jumprelu_encode = solutions.jumprelu_encode
decode_features = solutions.decode_features
feature_density = solutions.feature_density
l0 = solutions.l0
dead_feature_fraction = solutions.dead_feature_fraction
sae_variant_metrics = solutions.sae_variant_metrics
make_toy_superposition_batch = solutions.make_toy_superposition_batch
density_is_nondegenerate = solutions.density_is_nondegenerate
dictionary_recovery_report = solutions.dictionary_recovery_report
roc_auc_binary = solutions.roc_auc_binary
best_feature_auc = solutions.best_feature_auc
apply_decoder_steering = solutions.apply_decoder_steering
steering_comparison_report = solutions.steering_comparison_report
run_smoke_test = solutions.run_smoke_test


## Local Mechanics Tests

These tests cover encoder variants, reconstruction metrics, and the planted sparse-feature batch.


In [ ]:
tests.test_encoder_variants_match_reference_and_sparsity_rules(
    relu_l1_encode,
    topk_encode,
    gated_encode,
    jumprelu_encode,
)
tests.test_decode_and_metrics_match_identity_contract(
    decode_features,
    sae_variant_metrics,
    feature_density,
    l0,
    dead_feature_fraction,
)
tests.test_toy_superposition_batch_has_planted_sparse_structure(
    make_toy_superposition_batch,
    density_is_nondegenerate,
)


## Validation And Steering Tests

These tests cover duplicate dictionary recovery, held-out AUC, decoder-vector steering, and the whole smoke-test contract.


In [ ]:
tests.test_dictionary_recovery_detects_duplicates_and_missing_features(dictionary_recovery_report)
tests.test_best_feature_auc_handles_predictive_and_antipredictive_features(
    roc_auc_binary,
    best_feature_auc,
)
tests.test_decoder_steering_changes_last_position_and_reports_control(
    apply_decoder_steering,
    steering_comparison_report,
)
tests.test_notebook_contract(run_smoke_test)


## Signature Result

<details>
<summary>Interpreting the signature result</summary>

The learned SAE beats the zero baseline and the permuted decoder control on held-out activations. The best selected feature separates a generated technical/everyday split, and decoder-vector steering moves the matching projection score more than an orthogonal random direction. This does not make the feature name a proven semantic explanation.

</details>


In [ ]:
import json


def _load_committed_gpu_report() -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


def signature_table(gpu: dict) -> list[tuple[str, object]]:
    return [
        ("model", gpu["model_id"]),
        ("train / held-out prompts", f"{gpu['train_prompt_count']} / {gpu['heldout_prompt_count']}"),
        ("SAE", f"width {gpu['sae_width']}, TopK {gpu['sae_k']}, {gpu['training_steps']} steps"),
        ("held-out MSE", round(gpu["heldout_reconstruction_mse"], 4)),
        ("zero baseline MSE", round(gpu["zero_baseline_mse"], 4)),
        ("permuted decoder MSE", round(gpu["permuted_decoder_mse"], 4)),
        ("best feature AUC", round(gpu["best_feature_auc"], 4)),
        ("held-out L0", gpu["heldout_l0"]),
        ("dead feature fraction", round(gpu["heldout_dead_feature_fraction"], 4)),
        ("steer delta / random delta", f"{gpu['decoder_projection_steered_delta']:.3f} / {gpu['decoder_projection_random_delta']:.2e}"),
        ("safe token logit delta", f"{gpu['safe_logit_token']!r}: {gpu['safe_logit_delta']:.3f}"),
        ("peak VRAM GB", round(gpu["peak_vram_gb"], 3)),
    ]


gpu = run_gpu_test(max_vram_gb=24.0)
signature_table(gpu)


In [ ]:
import matplotlib.pyplot as plt

gpu = run_gpu_test(max_vram_gb=24.0)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))

axes[0].bar(
    ["zero", "learned", "permuted"],
    [gpu["zero_baseline_mse"], gpu["heldout_reconstruction_mse"], gpu["permuted_decoder_mse"]],
    color=["#94a3b8", "#2563eb", "#f97316"],
)
axes[0].set_title("Held-out reconstruction")
axes[0].set_ylabel("MSE, lower is better")

axes[1].bar(
    ["everyday", "technical"],
    [gpu["best_feature_negative_mean"], gpu["best_feature_positive_mean"]],
    color=["#94a3b8", "#16a34a"],
)
axes[1].set_title(f"Feature {gpu['best_feature_id']} AUC={gpu['best_feature_auc']:.3f}")
axes[1].set_ylabel("mean activation")

axes[2].bar(
    ["decoder", "random"],
    [gpu["decoder_projection_steered_delta"], gpu["decoder_projection_random_delta"]],
    color=["#7c3aed", "#94a3b8"],
)
axes[2].set_title("Steering control")
axes[2].set_ylabel("projection delta")

fig.tight_layout()
plt.show()


## Limitations

The report proves a scoped Pythia-70M TopK SAE preflight. It does not prove full SAE scaling behavior, released SAE artifact quality, generated-completion steering, or a robust semantic interpretation of feature 214.
